In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- HF_TOKEN (read-only) is needed only to download the gated bigcode/the-stack-dedup ---
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN").strip()
except Exception as e:
    raise SystemExit("HF_TOKEN secret not available. Open this notebook in the Kaggle editor, "
                     "Add-ons -> Secrets -> tick HF_TOKEN, then Save Version -> Save & Run All. "
                     f"({type(e).__name__})")
print("HF_TOKEN present")


In [ ]:
# --- Tokenizer: ~1 GB sample (MIT/Apache/BSD Python + FineWeb-Edu) -> our own 50k BPE ---
if not os.path.exists(f"{WORK}/tokenizer.json"):
    get_ipython().system("python -m vishalam.data.pretrain.shard sample --out /tmp/tok_sample.txt --python-mb 850 --fineweb-mb 150")
    get_ipython().system(f"python -m vishalam.tokenizer.bpe_train --input /tmp/tok_sample.txt --vocab-size 50304 --workers 4 --out {WORK}/tokenizer.json")
    os.remove("/tmp/tok_sample.txt")
print("tokenizer:", os.path.getsize(f"{WORK}/tokenizer.json"), "bytes")


In [ ]:
# --- Token shards, first half of the source files (~5.3B tokens, ~10.6 GB) ---
get_ipython().system(f"python -m vishalam.data.pretrain.shard build --tokenizer {WORK}/tokenizer.json "
                     f"--local-out {WORK} --work-dir /tmp/w --raw-dir /tmp/raw --py-files 0:0.5 --fw-files 0:0.5 "
                     f"--make-val --target-tokens 5.3e9 --max-hours {hours_left():.2f} --workers 4")
